# 03 — Evaluation on MATH-500 (reused for every arm)

Change the **config cell** (`MODEL_ID`, `ARM`) and rerun for each arm. Everything else stays identical,
so every arm is evaluated with exactly the same prompts, sampling settings and grader.

**Arm 0 (base model)** is run two ways:
- **zero-shot, chat format**: the same prompt every trained arm gets (the starting point)
- **few-shot, plain text**: 4 worked MATH-train examples, the fair way to measure what a *base* model knows

Trained arms (SFT, DPO, GRPO) only need the zero-shot run.

## 0. Environment (must run BEFORE importing vllm)

In [ ]:
import os
os.environ["VLLM_USE_FLASHINFER_SAMPLER"] = "0"     # image's nvcc is CUDA 12.x -> FlashInfer JIT fails
os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"  # run the engine inside the notebook process

%load_ext autoreload
%autoreload 2

import sys; sys.path.insert(0, ".")
import json, torch, vllm
from datasets import load_dataset
from vllm import LLM, SamplingParams

from posttrain.prompts import (build_prompt, stop_token_ids, select_fewshot_examples,
                               build_fewshot_prompt, FEWSHOT_STOP_STRINGS)
from posttrain.evaluate import run_generation, grade, summarize, print_summary, save_run
from posttrain import grader

## 1. Config — the only cell to change between arms

In [ ]:
MODEL_ID = "Qwen/Qwen3-0.6B-Base"   # later: path or Hub ID of the SFT / DPO / GRPO checkpoint
ARM = "arm0_base"                   # later: "armA_sft", "armB_dpo", ...
RUN_FEWSHOT = True                  # only for the base model

HF_RESULTS_REPO = "Maci10/dpo-king-eval-results"   # private dataset repo, one config per run
MAX_TOKENS = 1024
SEED = 0

GPU_NAME = torch.cuda.get_device_name(0)
META = {"model": MODEL_ID, "arm": ARM, "gpu": GPU_NAME, "vllm": vllm.__version__,
        "max_tokens": MAX_TOKENS, "temperature": 0.0, "seed": SEED}
print(META)

## 2. Data + grader check

In [ ]:
math500 = load_dataset("Maci10/dpo-king-math500", split="test")
print(len(math500), math500.column_names)

grader.run_tests(verbose=False)   # raises if the grader is broken
print("grader OK")

## 3. Load the model — run this cell ONCE per kernel session

In [ ]:
llm = LLM(
    model=MODEL_ID,
    dtype="bfloat16",
    max_model_len=4096,            # room for the few-shot prompt + 1024 generated tokens
    gpu_memory_utilization=0.85,
    enforce_eager=True,            # skip CUDA-graph compilation (simpler, slower)
    seed=SEED,
)
STOP_IDS = stop_token_ids(llm.get_tokenizer())
print("stop ids:", STOP_IDS)

- model = MODEL_ID : downloads  (first time) and loads the weights onto the GPU
- dtype = 'bfloat16' : the number format Qwen3 was trained in: half the memory of full precison with no meaningful accuray loss
- max_model_len = 4096 : the longest sequence (prompt + answer) the enginer can handle 4,096 leaves room for the long few-shot prompt plus 1,024 genreated tokens 

- gpu_memory_utilization = 0.85 : vLLm reserves 85% of GPU memory up front; about 1.2GB for the weights and the rest for KV_Cache

- stop tokens: vLLM does not remove special tokens from what the model generates, it still produces <|im_end|> when it is done but vLLM will hide that token when it turns the generated token IDs back into text.

- Stopping on token IDs checks the raw tokens as they’re generated, before any conversion to text, so it always works. stop_token_ids() just looks up the ID numbers for <|im_end|> (151645) and <|endoftext|> (151643). Your printout should show those two numbers.

## 4. Zero-shot, chat format (every arm)
Greedy decoding (temperature 0) → one deterministic answer per problem = **pass@1**.

- this is running the evaluation on the zero-shot evaluation 

In [ ]:
zs_params = SamplingParams(temperature=0.0, max_tokens=MAX_TOKENS, stop_token_ids=STOP_IDS, seed=SEED)
zs_prompts = [build_prompt(ex["problem"]) for ex in math500]

zs_outputs, zs_secs = run_generation(llm, zs_prompts, zs_params)
zs_records = grade(math500, zs_prompts, zs_outputs)
zs_summary = summarize(zs_records, seconds=zs_secs)
print_summary(zs_summary, f"{ARM} zero-shot")

## 5. Few-shot, plain text (base model only)
4 short worked examples from **MATH train** (never MATH-500), one per subject, fixed seed.
No chat tokens; generation stops when the model starts writing the next "Problem:".

- This is few-shot prompting, also called in-context learning. We take 4 worked examples from the training set and paste them into the prompt itself, right before the real question

- The model reads the examples, notices the pattern (“after Solution: comes a step-by-step answer ending in \boxed{}”), and continues it for the new problem. That’s something base models are good at, because predicting what comes next in a pattern is exactly what pretraining teaches.

In [ ]:
if RUN_FEWSHOT:
    sft_all = load_dataset("Maci10/dpo-king-sft_all", split="train")
    FEWSHOT = select_fewshot_examples(sft_all, k=4, seed=SEED)
    for ex in FEWSHOT:
        print(ex["subject"], ex["level"], "| idx", ex["idx"])
    print("\n----- example prompt -----\n")
    print(build_fewshot_prompt(math500[0]["problem"], FEWSHOT))

In [ ]:
if RUN_FEWSHOT:
    fs_params = SamplingParams(temperature=0.0, max_tokens=MAX_TOKENS, seed=SEED,
                               stop=FEWSHOT_STOP_STRINGS, stop_token_ids=STOP_IDS)
    fs_prompts = [build_fewshot_prompt(ex["problem"], FEWSHOT) for ex in math500]

    fs_outputs, fs_secs = run_generation(llm, fs_prompts, fs_params)
    fs_records = grade(math500, fs_prompts, fs_outputs)
    fs_summary = summarize(fs_records, seconds=fs_secs)
    print_summary(fs_summary, f"{ARM} few-shot (4-shot)")

## 6. Compare + look at failures

In [ ]:
import pandas as pd

runs = {"zero-shot": zs_summary}
if RUN_FEWSHOT:
    runs["few-shot"] = fs_summary

table = pd.DataFrame({
    name: {"accuracy": s["accuracy"], "has_box": s["has_box_rate"],
           "hit_max_tokens": s["hit_max_tokens_rate"], "mean_tokens": s["mean_tokens"],
           **{f"L{k}": v["acc"] for k, v in s["by_level"].items()}}
    for name, s in runs.items()
}).T
table

In [ ]:
# Read a few wrong answers. Is the model wrong, or did it ramble / never box an answer?
import random
recs = zs_records
wrong = [r for r in recs if not r["correct"]]
for r in random.Random(0).sample(wrong, k=min(3, len(wrong))):
    print(f"L{r['level']} {r['subject']} | gold {r['gold']!r} | extracted {r['extracted']!r} | "
          f"{r['n_tokens']} tok | {r['finish_reason']}")
    print(r["response"][-400:], "\n" + "-" * 70)

## 7. Save locally and push to the Hub
Each run becomes a config of one private dataset repo, e.g. `load_dataset(HF_RESULTS_REPO, "arm0_base_zeroshot")`.
**Do this before pausing the Space** — local files disappear on restart.

In [ ]:
save_run(zs_records, zs_summary, f"{ARM}_zeroshot", hub_repo=HF_RESULTS_REPO,
         meta={**META, "prompt_format": "chat_zeroshot"})

if RUN_FEWSHOT:
    save_run(fs_records, fs_summary, f"{ARM}_fewshot4", hub_repo=HF_RESULTS_REPO,
             meta={**META, "prompt_format": "plain_fewshot4",
                   "fewshot_idx": [ex["idx"] for ex in FEWSHOT]})

- The results here are zero-shot and few-shot prompts on the base model 
    - One problem with zero-shot was the models ability to have the boxed answer which one might reason that the base model might have a problem extracting the formatting. But in the few-shot example that gets thrown out the door and the model can extract the formatting much better. 
    - But with both models accuracy was very poor. 
    - Zero-shot showed a selection effect where the problems the model finished it mostly could solve but the ones it couldn't it rambled on and on ultimately not being able to solve them. 
    - Few-shot made the model finish everything the extra finished answers are mostly wrong. 
    - The real limitation in the base model is not formatting its the ablity to solve math problems, when we perform SFT this should dramatically solve the formatting issue.
    - One thing to note is that few-shot problems helped easier problems (L2 = Level 2 went from 32% to 48%) but hurt Level 4 (21% to 14%). The only guess i can make is that the 4 example solutions were short so the model copied that style and wrote too little on hard problems because the mean length went from 756 to 272.